In [1]:
from wizard import Node, Flow, InputField, RadioField, GoNext

In [2]:
profile_screen = Node(
    "profile_screen",
    "Profile",
    instructions="""Greet the user (If you have not greeted them before), introduce yourself, then ask the user for their name and address.""",
    fields=[
        InputField("tenant_name", "Tenant Name", "fills the name of the calling user/tenant"),
        InputField("tenant_address", "Tenant Address", "fills the address of the user/tenant")
    ],
    next="location_screen"
)

location_screen = Node(
    "location_screen",
    "Location",
    instructions="""Learn whether the issue is inside their home or outside""",
    fields=[
        RadioField("location", "Location", "Select the location", [
            "Inside home",
            "Outside home"
        ])
    ],
    next=lambda ctx: "inside_home_area_screen" if ctx["location"] == "Inside home" else "outside_home_area_screen"
)


inside_home_area_screen = Node(
    "inside_home_area_screen",
    "Inside Home Area",
    instructions="Ask the user about the area where the issue is",
    fields=[
        RadioField("area", "Area", "", [
            "Floors, Walls and Stairs",
            "Plumbing",
            "Doors, Locks and Windows",
            "Electrics",
            "Alarms & Door Entry",
            "Heating & Hot Water",
            "Empty Repair"
        ])
    ],
    next={
        "Floors, Walls and Stairs": "floors_walls_stairs_screen",
        "Plumbing": "plumbing_screen",
        # "Roof leaking": "roof_leaking_screen"
    }
)

floors_walls_stairs_screen = Node(
    "floors_walls_stairs_screen",
    "Area: Floors, walls and Stairs",
    "Ask the user about the precise area where the issue is",
    fields=[
        RadioField(
            "area_tier_2",
            "Which Area excatly has the problem?",
            "",
            [
                "Floors",
                "Ceilings",
                "Walls",
                "Stairs"
            ]
        )
    ],
    next={
        "Ceilings": "ceilings_issues_screen",
        "Floors": "floor_issues_screen"
    }
)

ceiling_issues_screen = Node(
    "ceilings_issues_screen",
    "Ceilings Issues",
    "Learn from the user which kindo of ceiling issue they area dealing with",
    fields=[
        RadioField("ceiling_issue", "Ceiling Issue", "", [
            "Ceiling is falling down",
            "Cracks in the ceiling",
            "Roof leaking"
    ])
    ],
    next={
        "Ceiling is falling down": "ceiling_is_falling_down_screen",
        "Cracks in the ceiling": "cracks_in_the_ceiling_screen",
        "Roof leaking": "roof_leaking_screen"
    }
)

cracks_in_the_ceiling_screen = Node(
    "cracks_in_the_ceiling_screen",
    "Cracks in the ceiling",
    "Learn whether the crack can fit a 1 euro coin or not",
    fields=[
        RadioField("crack_can_fit_coin", "Could you fit a one euro coin in the gap?", "", 
                   [
                       "yes",
                       "no"
                   ])
    ],
    next="exact_location_screen"
)

In [3]:
flow = Flow([profile_screen, 
             location_screen, 
             inside_home_area_screen, 
             ceiling_issues_screen, 
             floors_walls_stairs_screen, 
             ceiling_issues_screen, 
             cracks_in_the_ceiling_screen])

flow.play_actions(flow.current_action_model()())
# flow.play_actions([flow.current_action_model().__args__[0](value="Inside home")])
# flow.play_actions([GoNext()])
# flow.play_actions([flow.current_action_model().__args__[0](value="Floors, Walls and Stairs")])
# flow.play_actions([Next()])
# flow.play_actions([flow.current_action_model().__args__[0](value="Ceilings")])
# flow.play_actions([Next()])
# flow.play_actions([flow.current_action_model().__args__[0](value="Cracks in the ceiling")])
# flow.play_actions([Next()])




print(flow.render())


ValidationError: 3 validation errors for ActionModel
fill_tenant_name
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing
fill_tenant_address
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing
go_next
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing

In [4]:
flow.current_action_model()()

ValidationError: 3 validation errors for ActionModel
fill_tenant_name
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing
fill_tenant_address
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing
go_next
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing

In [ ]:
import warnings
warnings.filterwarnings("ignore")       

import os
from typing import Optional

import openai

from pydantic import BaseModel, Field

SYS = """
You are a customer support AI agent operating within our agent scripting system. Your role is to assist customers with their inquiries, issues, and requests in a helpful, professional, and efficient manner.

You will be given two pieces of information, event stream and agent script.

<event_stream>
This contains the conversation with the user so far.
</event_stream>

<agent_script>
contains the current node of the agent script, you can use the different actions to navigate across nodes or modify/input required data.
</agent_script>

<actions>
You are given actions that you can perform on the screen, you are also given actions to navigate nodes
Once you are done with a node, you can use the Next action to proceeed
</actions>

<rules>
- Each node comes with an instruction that you should follow
- Never provide inputs that the user has not provided!
- Take on action at a time
</rules>
""".strip()

async def call_llm(flow: Flow, event_stream: list):
    client = openai.AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

    class AgentOutput(BaseModel):
        response: str
        actions: Optional[flow.current_action_model()] = Field(..., 
                                                                     description="action to take given the current state (state = events stream and agent script)")
    
    event_stream_str = "\n".join(event_stream)
    user_msg = f"<event_stream>\n{event_stream_str}\n</event_stream>\n\n<agent_script>\n{flow.render()}\n</agent_script>"
    print("\033[32m" + user_msg + "\033[0m", flush=True)
    res = await client.beta.chat.completions.parse(
                model="gpt-4.1",
                messages=[
                    {
                        "role": "system",
                        "content": SYS,
                    },
                    {
                        "role": "user",
                        "content": user_msg,
                    },
                ],
                response_format=AgentOutput,
            )
    message = res.choices[0].message
    agent_output = message.parsed
    print(agent_output, flush=True)
    if agent_output.response:
        event_stream.append(f"Agent: {agent_output.response}")
    if agent_output.actions:
        for a, v in agent_output.actions:
            if v is not None:
                event_stream.append(f"Agent took Action: {v}")
        flow.play_actions(agent_output.actions)
    return agent_output


In [10]:
flow = Flow([profile_screen, 
             location_screen, 
             inside_home_area_screen, 
             floors_walls_stairs_screen, 
             ceiling_issues_screen, 
             cracks_in_the_ceiling_screen])

event_stream = []
action_taken = False
while True:
    if not action_taken:
        user_msg = input()
        if user_msg == "q":
            break
        event_stream.append(f"User: {user_msg}")
    agent_output = await call_llm(flow, event_stream)
    action_taken = bool(agent_output.actions)


<event_stream>
User: Hello
</event_stream

<agent_script>
Current Path: Profile

Node: Profile
Instructions: Greet the user (If you have not greeted them before), introduce yourself, then ask the user for their name and address.
---

Tenant Name (Input Field)
[...]
Tenant Address (Input Field)
[...]
</agent_script>
response="Hello! Welcome, and thank you for reaching out. I'm here to assist you today. Could you please provide me with your name and address to get started?" actions=None
<event_stream>
User: Hello
Agent: Hello! Welcome, and thank you for reaching out. I'm here to assist you today. Could you please provide me with your name and address to get started?
User: my name is Yasser Ahmed, my address is NY st. 123
</event_stream

<agent_script>
Current Path: Profile

Node: Profile
Instructions: Greet the user (If you have not greeted them before), introduce yourself, then ask the user for their name and address.
---

Tenant Name (Input Field)
[...]
Tenant Address (Input Field)
[..

CancelledError: 

In [10]:
flow.current_action_model()

typing.Union[wizard.FillTenantName, wizard.FillTenantAddress, wizard.Next]